# Customer Churn Prediction for Pakistani Telecom
## Exploratory Data Analysis & Model Training Notebook

This notebook provides exploratory data analysis (EDA), data cleaning, feature preprocessing, and initial machine learning model experimentation for the **Pakistan Telecom Dataset** from Kaggle.

In [ ]:
# Install required packages if running in Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q xgboost shap plotly scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries imported successfully.')

## 1. Data Loading and Quick Inspection

In [ ]:
# Load dataset
df = pd.read_csv('../data/churnData.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Summary information
df.info()

In [ ]:
# Check for missing values and duplicates
print("Missing Values per Column:")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

## 2. Target Variable Analysis

In [ ]:
# Identify target column
target_col = 'Churn'
print(df[target_col].value_counts(normalize=True) * 100)

# Plot distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col, palette='Set2')
plt.title('Customer Churn Distribution')
plt.xlabel('Churn Status')
plt.ylabel('Count')
plt.show()

## 3. Data Preprocessing & Model Training

In [ ]:
# Encode binary target
df['Churn_Encoded'] = (df[target_col].astype(str).str.strip().str.lower() == 'yes').astype(int)

# Separate X and y
X = df.drop(columns=[target_col, 'Churn_Encoded'])
y = df['Churn_Encoded']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Build preprocessor
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Processed Training Shape: {X_train_proc.shape}")
print(f"Processed Testing Shape: {X_test_proc.shape}")

In [ ]:
# Train Models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_proc, y_train)
    y_pred = model.predict(X_test_proc)
    y_prob = model.predict_proba(X_test_proc)[:, 1]
    
    results.append({
        'Model': name,
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

pd.DataFrame(results)